# 真实 Phase9G 残差训练与 Phase10D 比较

**实验已确认：**冻结 Phase9G，令 $r=D_{GT}-D_{Phase9G}$。训练普通 signed residual RF 与 HJD；与原 Phase10D-strict 同 validation600 比较。

这是**旧预测误差修正 pilot**，不是 water-dose reference、物理输运或盲 final-test 验证。旧结果与权重不覆盖。

默认从6个train cases各取32条（192条真实训练记录），先12条train-only可学习性检查，再进行两组fresh初始化pilot。不会称作全12,000条训练。

**执行顺序：P0 → P1 → P2 → P3 → P4 → P5a/P5b → P6 → P7。**新训练开关默认关闭，完成前置步骤后逐个开启。不要重跑旧S3/S4。


## P0 — 恢复完整代码并核对软件
不依赖旧Friday helpers。不升级已有PyTorch。14项新增测试使用临时软件数据，不评价你的模型；完整ZIP还保留原24项测试。

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, zipfile, sys, os, importlib.util, subprocess

PAYLOAD_B64 = '...'
PAYLOAD_SHA256 = '7c70b2c08ecd375d322ed7726cec626f31be103cd7e867f19f17f904572c64fe'
blob = base64.b64decode(PAYLOAD_B64)
if hashlib.sha256(blob).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError("Embedded source archive hash mismatch.")
TOOLS = Path("/content") / ("pg9_real_" + PAYLOAD_SHA256[:12])
TOOLS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    release = json.loads(archive.read("RELEASE.json"))
    for info in archive.infolist():
        relative = Path(info.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe archive path.")
        dest = TOOLS / relative
        data = archive.read(info.filename)
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and dest.read_bytes() != data:
            raise RuntimeError("Existing runtime code was modified: " + str(dest))
        if not dest.exists():
            dest.write_bytes(data)
    for name, expected in release["files"].items():
        if hashlib.sha256((TOOLS/name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Extracted source hash mismatch: " + name)

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Use a Colab runtime with PyTorch; do not silently install a different CUDA build.")
missing = [name for name in ["numpy", "pandas", "matplotlib", "pytest"]
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

# A same-runtime original recovery import is allowed ONLY if its source bytes match.
for pkg in ["p10recover", "pg9learn"]:
    if pkg in sys.modules:
        previous = Path(sys.modules[pkg].__file__).parent
        for name, expected in release["files"].items():
            rel = Path(name)
            if len(rel.parts)==2 and rel.parts[0]==pkg and rel.suffix==".py":
                candidate = previous/rel.name
                if not candidate.is_file() or hashlib.sha256(candidate.read_bytes()).hexdigest()!=expected:
                    raise RuntimeError("A different package version is already imported. Restart the runtime; do not mix models.")
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
from p10recover.verify import verify_source
verify_source()

# The subprocess returns pytest's real exit code. os._exit avoids unrelated C++
# interpreter shutdown waits after pytest has already completed its cleanup.
env = dict(os.environ, PYTHONPATH=str(TOOLS), PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
           OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MPLBACKEND="Agg")
program = "import pytest,os,sys; result=pytest.main(['-q',sys.argv[1],'--disable-warnings']); sys.stdout.flush();sys.stderr.flush();os._exit(int(result))"
p = subprocess.run([sys.executable,"-u","-c",program,str(TOOLS/"tests/test_signed_real.py")],
                   env=env, text=True, capture_output=True)
print(p.stdout)
if p.stderr:print(p.stderr)
if p.returncode:
    raise RuntimeError("New software tests failed. Do not bypass this error.")
BOOTSTRAP_TEST_LOG = p.stdout + p.stderr
print("TOOLS:", TOOLS)
print("Software verified; no user arrays or trained checkpoints were evaluated here.")


## P1 — 挂载并固定原结果与新输出位置
读取原run01_val600，不运行旧网络。CACHE_ID与RUN_ID不混用到不同recipe。GPU用于新训练；CPU可检查文件。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, torch, pandas as pd, numpy as np
from datetime import datetime, timezone
from p10recover.io import read_json, write_json
from pg9learn.common import runtime_settings, device_identity

ROOT = Path("/content/drive/MyDrive/rectified_flow_ct2dose")
OLD_RUN = Path("/content/drive/MyDrive/ct2dose_thesis/phase10d_recovered_v1/run01_val600")
EXP_ROOT = Path("/content/drive/MyDrive/ct2dose_thesis/phase9g_signed_real_v1")
CACHE_ID = "train192_seed42"
RUN_ID = "pilot_seed17"
EVAL_TAG = "eval01"  # New tag only for an intentionally separate evaluation.
CACHE_DIR = EXP_ROOT/"cache.local"/CACHE_ID
RUNS = EXP_ROOT/"runs.local"
if not ROOT.is_dir():raise FileNotFoundError(ROOT)
old_status = read_json(OLD_RUN/"run_status.json")
if old_status.get("status")!="completed_validation_inference" or old_status.get("n_records")!=600:
    raise RuntimeError("The old real val600 run must already be complete; do not use smoke4.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
runtime_settings(17)
EXP_ROOT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
write_json(EXP_ROOT/("environment_"+STAMP+".local.json"), device_identity(DEVICE))
(EXP_ROOT/("software_tests_"+STAMP+".log")).write_text(BOOTSTRAP_TEST_LOG,encoding="utf-8")
print("DEVICE:",DEVICE)
print("OLD_RUN:",OLD_RUN)
print("CACHE_DIR:",CACHE_DIR)
print("NEW_RUNS:",RUNS)
print("Old networks/results remain frozen. This is NOT a water-reference experiment.")


## P2 — 首次缓存时加载冻结的原系统
缓存READY已经存在时会跳过。首次确认这些是自己的原训练checkpoint后，将TRUST_OWN_CHECKPOINTS改为True。不会重跑旧val600，也不优化任何原模型。

In [ ]:
from p10recover.pipeline import LegacyPipeline
TRUST_OWN_CHECKPOINTS = False
FROZEN_PIPELINE = None
if (CACHE_DIR/"READY.json").is_file():
    print("Completed cache exists; old model loading is not needed for training from cache.")
else:
    if not TRUST_OWN_CHECKPOINTS:
        raise RuntimeError("Confirm your own original checkpoints and set TRUST_OWN_CHECKPOINTS=True in P2.")
    FROZEN_PIPELINE = LegacyPipeline(ROOT, device=DEVICE, trusted=True)
    # LegacyPipeline already applies strict binding and freezes its modules.
    print("Frozen Phase9G producer loaded. Only selected real training records will receive new cached predictions.")


## P3 — 192条真实train缓存 + 复用已有val600
首次将PREPARE_REAL_CACHE=True。选样只依赖case/record标识，不依赖target或误差。缓存生成可继续；不读取test JSON。完成后数据载入RAM，训练batch不再访问Drive小文件。

In [ ]:
from pg9learn.cache import prepare, Cache
PREPARE_REAL_CACHE = False
TRAIN_RECORDS_PER_CASE = 32
if PREPARE_REAL_CACHE and not (CACHE_DIR/"READY.json").is_file():
    if FROZEN_PIPELINE is None:
        raise RuntimeError("Run P2 with trusted original checkpoints first.")
    prepare(ROOT, OLD_RUN, CACHE_DIR, FROZEN_PIPELINE,
            n_per_case=TRAIN_RECORDS_PER_CASE,seed=42)
if not (CACHE_DIR/"READY.json").is_file():
    raise RuntimeError("Set PREPARE_REAL_CACHE=True in P3 to create the real training cache.")
cache = Cache(CACHE_DIR)
print("Training records:",len(cache.train),"Validation records:",len(cache.val))
print(json.dumps(cache.norm,indent=2))
audit = pd.read_csv(CACHE_DIR/"residual_audit.local.csv",dtype={"case_id":str})
print("Train records with both residual signs:",int(((audit.positive_fraction>0)&(audit.negative_fraction>0)).sum()))
print("Max decomposition error:",audit.reconstruction_max_abs.max())
print("Max target overlap:",audit.gt_overlap.max())
display(audit[["sample_id","residual_min","residual_max","positive_fraction","negative_fraction"]].head(12))
# Release old GPU modules after the immutable training cache is ready.
if FROZEN_PIPELINE is not None:
    for name in ["base_model","model_9d_plus","phase10d_model"]:
        model = getattr(FROZEN_PIPELINE.legacy,name,None)
        if model is not None:model.to("cpu")
    FROZEN_PIPELINE = None
    if torch.cuda.is_available():torch.cuda.empty_cache()
print("READY: cached real Phase9G-error targets; train-only scaling; no new reference field.")


## P4 — 先做12条真实train-only可学习性检查
RUN_SMALL=True启动两组各128updates。所有监控也只用这12条train。它不是泛化评价；pilot将重新初始化。若未改善零修正，保留结果，先暂停该分支，不改旧reference或使用真实mass作弊。

In [ ]:
from pg9learn.train import train, recipe
RUN_SMALL = False
SMALL_RUNS = {m:RUNS/("small_"+m+"_seed17") for m in ["residual_rf","hjd_rf"]}
if RUN_SMALL:
    for method, path in SMALL_RUNS.items():
        train(cache,method,path,recipe("small"),DEVICE)
rows=[]
for method,path in SMALL_RUNS.items():
    f=path/"training_summary.json"
    if f.is_file():
        s=read_json(f)
        rows.append(dict(method=method,train_records=12,
                         initial_rmse=s["initial_monitor"]["rmse"],
                         best_train_rmse=s["best_monitor"]["rmse"],
                         phase9g_rmse=s["best_monitor"]["base_rmse"],
                         learned_beyond_zero_correction=s["learned_beyond_zero_correction"]))
if rows:display(pd.DataFrame(rows))
else:print("Small runs not yet executed; set RUN_SMALL=True.")


## P5a — 普通signed residual RF pilot
真实192train、384updates；固定40validation选best。先读P4结果。修改recipe后需要新版本/run，不覆盖旧checkpoint。

In [ ]:
RUN_RF_PILOT = False
RF_RUN = RUNS/(RUN_ID+"_residual_rf")
if RUN_RF_PILOT:
    small_status=read_json(SMALL_RUNS["residual_rf"]/"run_status.json")
    if small_status.get("status")!="completed_training":raise RuntimeError("Complete the RF small-train check first.")
    train(cache,"residual_rf",RF_RUN,recipe("pilot"),DEVICE)
if (RF_RUN/"training_summary.json").is_file():
    print(json.dumps(read_json(RF_RUN/"training_summary.json"),indent=2))
else:print("RF pilot not yet completed.")


## P5b — HJD两条空间flow + mass heads pilot
与普通RF同train/monitor/更新预算，但网络、loss、计算成本不同。不是字面复制PDF：uniform source、条件编码、mass heads与endpoint辅助loss是明确实现选择。

In [ ]:
RUN_HJD_PILOT = False
HJD_RUN = RUNS/(RUN_ID+"_hjd_rf")
if RUN_HJD_PILOT:
    small_status=read_json(SMALL_RUNS["hjd_rf"]/"run_status.json")
    if small_status.get("status")!="completed_training":raise RuntimeError("Complete the HJD small-train check first.")
    train(cache,"hjd_rf",HJD_RUN,recipe("pilot"),DEVICE)
if (HJD_RUN/"training_summary.json").is_file():
    print(json.dumps(read_json(HJD_RUN/"training_summary.json"),indent=2))
else:print("HJD pilot not yet completed.")


## P6 — 新模型val600预测与相同旧指标比较
只预测两组新模型；原Phase9G/Phase10D等结果读取已有文件，不再运行原网络。保存raw/clamped与HJD原始分量。每条新记录可恢复，两个VM不能同时写同一目录。

In [ ]:
from pg9learn.compare import evaluate_new, report
RUN_COMPARE600 = False
EVALUATIONS = {
    "residual_rf":EXP_ROOT/"evaluations.local"/(RUN_ID+"_residual_rf_"+EVAL_TAG),
    "hjd_rf":EXP_ROOT/"evaluations.local"/(RUN_ID+"_hjd_rf_"+EVAL_TAG),
}
if RUN_COMPARE600:
    for path in [RF_RUN,HJD_RUN]:
        if read_json(path/"run_status.json").get("status")!="completed_training":
            raise RuntimeError("Both new pilots must finish before the combined val600 comparison.")
    evaluate_new(cache,RF_RUN,EVALUATIONS["residual_rf"],DEVICE)
    evaluate_new(cache,HJD_RUN,EVALUATIONS["hjd_rf"],DEVICE)
    report_stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    REPORT=report(cache,list(EVALUATIONS.values()),EXP_ROOT/"reports.local"/report_stamp)
    write_json(EXP_ROOT/"LATEST_REPORT.local.json",{"path":str(REPORT)},replace=True)
    print("REPORT:",REPORT/"report.html")
else:
    marker=EXP_ROOT/"LATEST_REPORT.local.json"
    REPORT=Path(read_json(marker)["path"]) if marker.is_file() else None
    print("No new inference. Existing report:",REPORT)


## P7 — 读取结果与小型报告打包
旧模型、ordinary RF和HJD必须在同一cohort上评分。negative delta表示降低误差，百分指标delta单位是百分点。新结果即使更差也保留，不自动改参数使其获胜。

In [ ]:
if REPORT is None or not (REPORT/"REPORT_READY.json").is_file():
    print("No completed new-method comparison report yet.")
else:
    files_manifest=read_json(REPORT/"FILES.json")
    for name,expected in files_manifest.items():
        if hashlib.sha256((REPORT/name).read_bytes()).hexdigest()!=expected:
            raise RuntimeError("Report file changed: "+name)
    table=pd.read_csv(REPORT/"comparison.csv")
    with pd.option_context("display.float_format",lambda x:f"{x:.8g}"):
        display(table[["method","rmse_stored_units","mae_stored_units","x_mean_pct","y_mean_pct","z_mean_pct","n_cases","n_records"]])
        display(pd.read_csv(REPORT/"paired_deltas_cases.local.csv",dtype={"case_id":str}))
        display(pd.read_csv(REPORT/"new_method_resources.csv"))
    print("Report HTML:",REPORT/"report.html")
    print(json.dumps(read_json(REPORT/"REPORT_READY.json"),indent=2))

EXPORT_SMALL_REPORT = False
if EXPORT_SMALL_REPORT and REPORT is not None:
    from google.colab import files as colab_files
    package=REPORT/"private_small_report.zip"
    # No arrays, medical figures, source credentials, or trained checkpoints.
    names=["comparison.csv","paired_deltas_cases.local.csv","new_method_resources.csv", "report.md","REPORT_READY.json"]
    with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED) as archive:
        for name in names:archive.write(REPORT/name,arcname=name)
        for method,run in [("residual_rf",RF_RUN),("hjd_rf",HJD_RUN)]:
            for name in ["training_summary.json","history.csv"]:
                archive.write(run/name,arcname=method+"/"+name)
    print("Private summary only; case identifiers may remain. Do not publish blindly.")
    colab_files.download(str(package))


## 恢复与范围

断线后确认旧writer结束，运行P0/P1；缓存READY存在可跳过P2，P3只读缓存。运行原P4/P5会从`last.json`核验并恢复；已完成不会重复训练。最后未保存的update可能重算。

源码、数据、配置、训练环境不同会拒绝精确续训；不要删除检查。CUDA grid sampling等可能仍非逐位确定。原test不参与；旧上游暴露不被这轮冻结消除。

本轮完成是“真实旧预测残差修正的开发比较”；不是物理water-reference研究完成。
